In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns',100)
pd.set_option('display.width',200)
DATA_PATH = "../data/raw/diabetic_data.csv"
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()


Shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [5]:
print("Data types breakdown:")
print(df.dtypes.value_counts())

Data types breakdown:
str      37
int64    13
Name: count, dtype: int64


**Observation** 37 columns are stored as 'object' (text/categorical) and 13 as 'int64'. Zero float columns, this means the values were already discretized, raw values were not given.


In [12]:
# This dataset uses '?' as the missing-value marker, not NAN typically seen in class
missing_count = (df =='?').sum()
missing_pct = ((df == '?').mean() * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
    })
missing_summary[missing_summary['missing_count'] > 0]
missing_summary = missing_summary.sort_values('missing_count', ascending = False)

missing_summary

,missing_count,missing_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02
admission_type_id,0,0.00
patient_nbr,0,0.00
encounter_id,0,0.00


In [13]:
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
            'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
            'miglitol', 'troglitazone', 'tolazamide', 'examide',
            'citoglipton', 'insulin', 'glyburide-metformin',
            'glipizide-metformin', 'glimepiride-pioglitazone',
            'metformin-rosiglitazone', 'metformin-pioglitazone']

#percent of rows in the most common category
near_constant = []
for col in med_cols:
    top_pct = df[col].value_counts(normalize = True).iloc[0]
    near_constant.append({
            'medication': col,
            'most_common_pct': round(top_pct * 100, 2)
        })

    med_summary = pd.DataFrame(near_constant).sort_values(
        'most_common_pct', ascending=False)
med_summary

,medication,most_common_pct
5,acetohexamide,100.00
15,examide,100.00
20,glimepiride-pioglitazone,100.00
22,metformin-pioglitazone,100.00
16,citoglipton,100.00
21,metformin-rosiglitazone,100.00
13,troglitazone,100.00
19,glipizide-metformin,99.99
8,tolbutamide,99.98
14,tolazamide,99.96


## Medication Columns: Near-Constant Features Identified ##

The dataset contains 23 medication columns, I computed the percentage of rows in the most common category for each column.  15 of the columns have greater than 99% in a single category. The most variable columns are insulin, metformin and glipizide. There is evidence to support the possible dropping of the near constant values before modeling. This is due to the fact that features with no variation can't contribute to the predictive signal. Deferring the decision for now.

In [14]:
# Count how many columns are near-constant at the 99% threshold
threshold = 99.0
near_constant_count = (med_summary['most_common_pct'] >= threshold).sum()
total_meds = len(med_summary)

print(f"Out of {total_meds} medication columns:")
print(f"  - {near_constant_count} are near-constant (≥{threshold}% same value)")
print(f"  - {total_meds - near_constant_count} have meaningful variation")
print()
print("The near-constant columns:")
print(med_summary[med_summary['most_common_pct'] >= threshold])

Out of 23 medication columns:
  - 15 are near-constant (≥99.0% same value)
  - 8 have meaningful variation

The near-constant columns:
                  medication  most_common_pct
5              acetohexamide           100.00
15                   examide           100.00
20  glimepiride-pioglitazone           100.00
22    metformin-pioglitazone           100.00
16               citoglipton           100.00
21   metformin-rosiglitazone           100.00
13              troglitazone           100.00
19       glipizide-metformin            99.99
8                tolbutamide            99.98
14                tolazamide            99.96
12                  miglitol            99.96
3             chlorpropamide            99.92
11                  acarbose            99.70
2                nateglinide            99.31
18       glyburide-metformin            99.31


In [15]:
unique_counts = df.nunique().sort_values()
print(f"Total columns: {len(unique_counts)}")
print()
print("Columns with very few unique values (<=3) - possibly constant:")
print(unique_counts[unique_counts <=3])
print()
print("High-cardinality columns (>500 unique values):")
print(unique_counts[unique_counts > 500])

Total columns: 50

Columns with very few unique values (<=3) - possibly constant:
citoglipton                 1
examide                     1
acetohexamide               2
glipizide-metformin         2
tolbutamide                 2
troglitazone                2
metformin-rosiglitazone     2
glimepiride-pioglitazone    2
metformin-pioglitazone      2
change                      2
diabetesMed                 2
gender                      3
tolazamide                  3
readmitted                  3
max_glu_serum               3
A1Cresult                   3
dtype: int64

High-cardinality columns (>500 unique values):
diag_1             717
diag_2             749
diag_3             790
patient_nbr      71518
encounter_id    101766
dtype: int64


## Cardinality Observations ##

** Constant / near- constant columns:**
'examide' and 'citoglipton' are completely constant across all rows. These contribute zero predictive signal and will be dropped before modeling.

'encounter_id' (101,766) and 'patient_nbr are identifiers, not features. The will be dropped from modeling after deduplication


In [20]:
# Format numbers to 2 decimals, no scientific notation
pd.set_option('display.float_format', '{:.2f}'.format)

# Exclude ID columns from the numeric summary — their stats are meaningless
id_cols = ['encounter_id', 'patient_nbr']
numeric_cols = df.select_dtypes(include='number').columns.difference(id_cols)

df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
admission_source_id,101766.00,5.75,4.06,1.00,1.00,7.00,7.00,25.00
admission_type_id,101766.00,2.02,1.45,1.00,1.00,1.00,3.00,8.00
discharge_disposition_id,101766.00,3.72,5.28,1.00,1.00,1.00,4.00,28.00
num_lab_procedures,101766.00,43.10,19.67,1.00,31.00,44.00,57.00,132.00
num_medications,101766.00,16.02,8.13,1.00,10.00,15.00,20.00,81.00
num_procedures,101766.00,1.34,1.71,0.00,0.00,1.00,2.00,6.00
number_diagnoses,101766.00,7.42,1.93,1.00,6.00,8.00,9.00,16.00
number_emergency,101766.00,0.20,0.93,0.00,0.00,0.00,0.00,76.00
number_inpatient,101766.00,0.64,1.26,0.00,0.00,0.00,1.00,21.00
number_outpatient,101766.00,0.37,1.27,0.00,0.00,0.00,0.00,42.00


## Numeric Feature Distributions

**Initial observations on the numeric columns:**

- `encounter_id` and `patient_nbr` appeared in `.describe()` because they're stored 
  as integers, but their statistics are meaningless (they're identifiers). Excluded 
  from this analysis.
- `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are 
  also stored as integers but represent **categorical codes** — not true numerics. 
  These will be one-hot encoded as categoricals, not standardized.

**Genuine numeric features and their behavior:**
- `time_in_hospital`: mean ~4.4 days, std 3.0, range likely 1–14. Reasonable scale.
- `num_lab_procedures`: mean ~43, std ~20. Wide range — will benefit from standardization.
- `num_medications`: mean ~16, std ~8. Moderate range.
- `number_diagnoses`: mean ~7.4, std ~1.9. Tight range, low variance.
- `number_outpatient`, `number_emergency`, `number_inpatient`: heavily right-skewed 
  — std is 3-5× the mean and min is 0, indicating most patients have zero prior 
  visits while a small subset has many. **This is exactly the pattern expected 
  for prior-utilization features**, and these are typically among the strongest 
  predictors of readmission.

**Decisions (deferred to feature engineering):**
- Apply `StandardScaler` to genuine numeric features before logistic regression
- Apply `np.log1p()` transformation to the right-skewed visit-count features 
  (`number_outpatient`, `number_emergency`, `number_inpatient`) to compress the 
  long tail before scaling
- One-hot encode the integer-coded categoricals (`admission_type_id`, etc.) 
  rather than treating them as numeric

In [21]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_parquet('../data/processed/cohort_raw.parquet')

print(f"Saved {len(df):,} rows x {df.shape[1]} colums to:")
print(f" ../data/processed/cohort_raw.parquet")

Saved 101,766 rows x 50 colums to:
 ../data/processed/cohort_raw.parquet
